# The Simpsons
By: pierremegret

## Download Dataset from Kaggle

In [3]:
# source: https://www.kaggle.com/discussions/general/74235
from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

#! kaggle kernels pull ambarish/fun-in-text-mining-with-simpsons
!kaggle datasets download -d pierremegret/dialogue-lines-of-the-simpsons

! unzip "*.zip"

Dataset URL: https://www.kaggle.com/datasets/pierremegret/dialogue-lines-of-the-simpsons
License(s): CC-BY-SA-3.0
100% 3.32M/3.32M [00:01<00:00, 2.42MB/s]

Archive:  dialogue-lines-of-the-simpsons.zip
  inflating: simpsons_dataset.csv    


In [4]:
import re  # For preprocessing
import pandas as pd  # For data handling
from time import time  # To time our operations
from collections import defaultdict  # For word frequency

import spacy  # For preprocessing

import logging  # Setting up the loggings to monitor gensim
logging.basicConfig(format="%(levelname)s - %(asctime)s: %(message)s", datefmt= '%H:%M:%S', level=logging.INFO)


In [5]:
df = pd.read_csv('simpsons_dataset.csv')
df.shape
df.head()

,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [7]:
df = df.dropna().reset_index(drop=True)
df.isnull().sum()

,0
raw_character_text,0
spoken_words,0


In [9]:
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser']) # disabling Named Entity Recognition for speed

def cleaning(doc):
    # Lemmatizes and removes stopwords
    # doc needs to be a spacy Doc object
    txt = [token.lemma_ for token in doc if not token.is_stop]

    # Word2Vec uses context words to learn the vector representation of a target word,
    # the benefit for the training is very small
    if len(txt) > 2: # if a sentence is only one or two words long,
        return ' '.join(txt)

In [12]:
# removes non-alphanumeric characters
brief_cleaning = (re.sub("[^A-Za-z']+", ' ', str(row)).lower() for row in df['spoken_words'])

# using pipe method
t = time()
txt = [cleaning(doc) for doc in nlp.pipe(brief_cleaning, batch_size=5000)]
print('Time to clean up everything: {} mins'.format(round((time() - t) / 60, 2)))

Time to clean up everything: 2.35 mins


Okay, now let's see how well a Naive Bayes classifier can do by just looking at the words in the randomly chosen tweet.

In [ ]:
# pull the data into vectors
vectorizer = CountVectorizer()
x = vectorizer.fit_transform(twigen_confident['text_norm'])

encoder = LabelEncoder()
y = encoder.fit_transform(twigen_confident['gender'])
print(encoder.classes_)

# split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# take a look at the shape of each of these
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)


# Training

In [ ]:
nb = MultinomialNB()
nb.fit(x_train, y_train)

print(nb.score(x_test, y_test))

So we get about 62% accuracy on the "best" observations, using only tweet text.

Let's try a couple more features. Specifically, let's add the description text by concatenating it to the tweet text.

In [ ]:
twigen_confident['all_features'] = twigen_confident['text_norm'].str.cat(twigen_confident['description_norm'], sep=' ')
twigen_confident.head()
#twigen_confident = twigen[twigen['gender:confidence']==1]

In [ ]:
# pull the data into vectors
vectorizer = CountVectorizer()
x = vectorizer.fit_transform(twigen_confident['all_features'])

encoder = LabelEncoder()
y = encoder.fit_transform(twigen_confident['gender'])
print(encoder.classes_)

# split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

nb = MultinomialNB()
nb.fit(x_train, y_train)

print(nb.score(x_test, y_test))

Cool, so we gain about 2-3 percentage points in accuracy just by adding description text alongside tweet text.

You can use this kind of procedure to play around with adding more features, or try a different type of model and see how accurately you can predict gender. (Maybe also try including the less-confident observations; my exclusion of them was probably anti-conservative)

# Visualizations and Reports

In [ ]:
predicted = nb.predict(x_test)

print(
    f"Classification report for Kaggle dataset:\n"
    f"{metrics.classification_report(y_test, predicted, target_names=encoder.classes_)}\n"
)

In [ ]:
disp = metrics.ConfusionMatrixDisplay.from_predictions(
    y_test, predicted, display_labels=encoder.classes_)
disp.figure_.suptitle("Confusion Matrix")

plt.show()